# Surface Code Decoding

In this notebook, we aim to develop a minimum-weight decoder for nine-qubit rotated surface code. We will begin with creating binary representations (vectors) of stabilizers and logical operators. Then, we will build decoding graphs and evaluate recovery for every $Z$-error pattern.

Now, the current model uses $Z$-only errors and syndrome measurements. Each qubit independently has a $Z$-error with probability $p$. Next, we are going to reveal circuit-based syndrome extraction in Qiskit.

**Content**

1. Define the code and binary representation
2. Identify stabilizers and logical operators
3. Construct the decoding graph
4. Generate all Z-errors
5. Build the minimum-weight decoder table
6. Evaluate recovery and logical failure probability

## Define the code and binary representation

We are going to work on nine-qubit rotated surface code. The data qubits $q_i$ are placed on a physical patch as follows:

\\begin{matrix}q_0&q_1&q_2\\q_3&q_4&q_5\\q_6&q_7&q_8\end{matrix}

Suppose there is a $Z$-error on a qubit $j$. It means that the Pauli-$Z$ operator acts on the qubit $q_j$. In this case, we represent the error by an error vector $e$ such that $e_j = 1$. For example, suppose that there is an error string `ZZIZIIIII`. Then, the binary representation is $e = (\begin{matrix}1 & 1 & 0&1&0&0&0&0&0\end{matrix})$

$X$ and $Z$-type stabilizer generators are represented by matrices $H_x$ and $H_z$. Multiplying of same type operators corresponds to adding support vectors modulo two. A $Z$- and $X$-error *syndrome* is computed as follows:

$$ s_z = H_X e \text{ (mod 2)} $$
$$ s_x = H_Z e \text{ (mod 2)} $$

Let us remind of syndromes Given a quantum state $\ket{\psi}$, assume that $E$ is error and $S$ denotes the stabilizer. Then,

$$ ES\ket{\psi} = (\pm 1) SE \ket{\psi}$$ Since $S\ket{\psi} = \ket{\psi}$,
$$ ES\ket{\psi} = E \ket{\psi} = (\pm 1) SE \ket{\psi}$$ So, $\pm 1$ gives eigenvalues of $S$. We call these eigenvalues *syndrome*. So the syndrome vectors, $s_x$ and $s_z$, consist of these eigenvalues. However, we store $0$ and $1$ in the vectors instead of $\pm 1$. A syndrome bit of $1$ corresponds to the eigenvalue $-1$ while $0$ records the eigenvalue $+1$.




Given the $X$- and $Z$-type stabilisers $$ S_X = <XXIIIIIII, IXXIXXIII, IIIXXIXXI, IIIIIIIXX> $$
$$ S_Z = <ZZIZZIIII, IIZIIZIII, IIIZIIZII, IIIIZZIZZ> $$

let us view the stabiliser check matrices $H_X$ and $H_Z$:

In [2]:
import numpy as np

H_x = np.array([
    [1, 1, 0, 0, 0, 0, 0, 0, 0], # XXIIIIIII
    [0, 1, 1, 0, 1, 1, 0, 0, 0], # IXXIXXIII
    [0, 0, 0, 1, 1, 0, 1, 1, 0], # IIIXXIXXI
    [0, 0, 0, 0, 0, 0, 0, 1, 1], # IIIIIIIXX
], dtype=int)

H_z = np.array([
    [1, 1, 0, 1, 1, 0, 0, 0, 0], # ZZIZZIIII
    [0, 0, 1, 0, 0, 1, 0, 0, 0], # IIZIIZIII
    [0, 0, 0, 1, 0, 0, 1, 0, 0], # IIIZIIZII
    [0, 0, 0, 0, 1, 1, 0, 1, 1], # IIIIZZIZZ
], dtype=int)

All X-type and Z-type generators commute.


Note that inner product of two vectors of $H_x$ and $H_z$ is a mathematical representation of product of two stabilisers consisting of Pauli strings. Remind that the number of operators that overlap at the same position is even, then the two operators commute. Otherwise, they anticommute with each other. In the mathematical way, we check the commutation looking at the inner product modulo $2$ of the two representations. If the product is $0$, then we understand that the number of the number of errors at the same position is even. The inner product result $1$ says that the number of overlapping operators is odd. Thus, we can express that

if $ <g_X, g_Z> = 1 $, then two generators $g_X$ and $g_Z$ anticommute

if $ <g_X, g_Z> = 0 $, then two generators $g_X$ and $g_Z$ commute

We can check all type generators through the check matrices $H_X$ and $H_Z$:

In [3]:
# checking commutation
assert np.all((H_x @ H_z.T) % 2 == 0)
print("All X-type and Z-type generators commute.")

All X-type and Z-type generators commute.


## Logical operators

A **logical operator** is a Pauli operator which commutes with all stabilisers but not stabiliser. Reminding the definition of normaliser of stabiliser

$$ \mathcal{N(S)} = \{P \in \mathcal{P}_n \quad | \quad SP = PS, \quad \forall S \in \mathcal{S}\}$$

So, if $P$ is logical operator, then $P \in \mathcal{N(S)} / \mathcal{S}$.

Let $S'$ be a stabiliser. Then, $PS'$ commutes with all stabilisers because $(PS')S = P(SS') = (PS)S' = (SP)S' = S(PS') $. Also, $PS'$ is not a stabiliser because $P \notin \mathcal{S}$. It follows that $PS' = L \in \mathcal{N(S)/S}$ is a logical operator.

Now let us build the logical operators and their relation with check matrices through Python.

Note that stabiliser generators belong to the *row space* of the check matrices $H_X$ and $H_Z$. Also, since logical operators commute with all stabilisers by the definition, inner product of all elements of the row space and logical operators should be $0$. Therefore, an $X$-type logical operator, let's denote $\mathcal{l}_X$, can be defined as

$$ H_Z \mathcal{l}_X  = 0, \text{\quad} \mathcal{l}_X \notin rowspan(H_X)$$

By the similar manner, a $Z$-type logical operator $\mathcal{l}_Z$ is defined as follows:

$$ H_X \mathcal{l}_Z  = 0, \text{\quad} \mathcal{l}_Z \notin rowspan(H_Z)$$

All of row vectors in the row space should be independent.

In [5]:
# generating all binary sums of the matrix rows, representing products of generators
def row_space(check_matrix):
    # assign the number of physical qubits
    n = check_matrix.shape[1]
    # initialize the span
    span = {tuple([0] * n)}

    for row in check_matrix:
        new_vectors = set()

        for vector in span:
            # check linear independence
            vector = (vector + row) % 2
            new_vectors.add(tuple(vector))

        span.update(new_vectors)

    return span

A stabiliser should be in a row space of check matrix

In [6]:
# check whether a candidate belongs to the row space
def check_stabilizer(stab_matrix, stabilizer):
    return tuple(stabilizer) in row_space(stab_matrix)

Now, here we check whether a candidate operator commutes with generators or not. The condition `if not check_stabilizer` says that the operator should not be a stabiliser. `np.vdot(logical_op, stabilizer) %2 != 0:` control the commutation of operator with the stabilisers. If the function pass the two conditions, it returns `True`, which means the candidate is a logical operator.

In [15]:
# check whether a candidate commutes with the opposite-type generators and is outside its own stabilizer row space
def check_logical_operator(check_matrix, logical_op, stab_matrix):
    # is it outside the stabilizer row space
    if not check_stabilizer(stab_matrix, logical_op):
        for stabilizer in check_matrix:
            # check commuting with the generators
            if np.vdot(logical_op, stabilizer) %2 != 0:
                return False
            else:
                continue
        return True
    else:
        return False


### Recovery operation

Let $e$ denotes the actual $Z$ error and $c$ is the $Z$ correction. The *residual* is the sum of $e$ and $c$ modulo 2:

$$ r = (e + c) \text{ mod(2)} $$

A residual syndrome is computed by the product of stabiliser matrix $H_X$ (we assume that we have $Z$-error) and $r$: $H_X r$ A zero residual syndrome places the quantum state back in the code space. More precisely, $H_X r = 0$ means that the quantum state the recovery has acted is in the code space. In this condition, there are two possibilities:

* $r$ represents $X$-type stabiliser: Suppose $R$ denotes correction, $E$ is error. $RE \in \mathcal{S}$, so $RE \ket{\psi} = \ket{\psi}$ for every encoded state. Recovery succeeds.
* $r$ represents logical $X$ operator: $RE$ is a nontrivial logical operator. The state remains in $C$, but encoded information can change. So, recovery fails

Logical flip we have defined as $f = \mathcal{l}_X.r$ gives information about what $r$ is. Given a logical representation $\mathcal{l}_X$, $f = \mathcal{l}_X.r = 0$ says that the $r$ commutes with any logical operator. By the definition, $r$ is stabiliser representation. Otherwise, $f = \mathcal{l}_X.r = 1$ (logical flip is 1) means that $r$ anticommutes with chosen logical $X$ operator, so $r$ is represented by a logical operator.

* $f = \mathcal{l}_X.r = 0$: the residual commutes with logical operator $L_X$. It is $Z$-type stabiliser. Recovery succeeds.
* $f = \mathcal{l}_X.r = 1$: the residual anticommutes with logical operator $L_X$. It is nontrivial logical operator. Recovery fails.

That's why we implement the checking expression: `bool(np.all(residual_syndrome == 0) and logical_flip == 0)`

In [7]:
def recovery(H_check, r, logical_op, stab_matrix):

    if not check_logical_operator(H_check, logical_op, stab_matrix):
        return None

    # is the state in the code space
    residual_syndrome = (stab_matrix @ r) % 2
    # is the r stabiliser
    logical_flip = (logical_op @ r) % 2

    # the recovered state should be in the code space as unchanged
    return bool(np.all(residual_syndrome == 0) and logical_flip == 0)

**Example**

Suppose that there are two $Z$-errors on the qubit $0$ and $1$ of a quantum state with $9$ physical qubits. So, we have an error operator $E = ZZIIIIIII$. Let the correction be $R = IIIIIZIII$. We are going to view a recovery operation.

Firstly, we need to convert the Pauli operators into binary vectors:

In [8]:
def error_to_vector(error):
    return np.array([int(symbol == "Z") for symbol in error], dtype=int)

We compute syndrome vectors by the matrix multiplication modulo $2$, which means matrix multiplication under the field $\mathbb{F}_2$:

In [ ]:
def compute_syndrome(H_x, error):
    return tuple(((H_x @ error_to_vector(error)) % 2).tolist())

We build the error and correction vectors:

In [33]:
# error Z0 Z1, correction Z5
e = error_to_vector("ZZIIIIIII")
c = error_to_vector("IIIIIZIII")

print(e)
print(c)

[1 1 0 0 0 0 0 0 0]
[0 0 0 0 0 1 0 0 0]


The sum of $e$ and $c$ is residual vector: $r = (e + c) \text{ mod2}$

In [34]:
r = (e + c) % 2
print(r)

[1 1 0 0 0 1 0 0 0]


Let the $X$- and $Z$-type stabilisers are given as:

$$ S_X = <XXIIIIIII, IXXIXXIII, IIIXXIXXI, IIIIIIIXX> $$
$$ S_Z = <ZZIZZIIII, IIZIIZIII, IIIZIIZII, IIIIZZIZZ> $$

Then, the check matrices consisting of these generator represenation vectors are

$$ H_X = \begin{pmatrix} 1 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
    0 & 1 & 1 & 0 & 1 & 1 & 0 & 0 & 0 \\
    0 & 0 & 0 & 1 & 1 & 0 & 1 & 1 & 0 \\
    0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 1 \end{pmatrix} $$


$$ H_Z = \begin{pmatrix} 1 & 1 & 0 & 1 & 1 & 0 & 0 & 0 & 0 \\
    0 & 0 & 1 & 0 & 0 & 1 & 0 & 0 & 0 \\
    0 & 0 & 0 & 1 & 0 & 0 & 1 & 0 & 0 \\
    0 & 0 & 0 & 0 & 1 & 1 & 0 & 1 & 1 \end{pmatrix} $$

In [35]:
print("Residual syndrome", ((H_x @ r) % 2).tolist())

Residual syndrome [0, 0, 0, 0]


The residual syndrome is zero vector. But, we need to look at logical flip as well. We need to check $f = \mathcal{l}_X r \text{ mod2}$.

Now, we need to a logical operator. Consider the operator $L_X = XIIXIIXII$. Is it logical operator? Let us check it:

In [27]:
def x_op_to_vector(error):
    return np.array([int(symbol == "X") for symbol in error], dtype=int)

logical_x = x_op_to_vector("XIIXIIXII")
check_logical_operator(H_z, logical_x, H_x)

True

So, $L_X$ is a logical operator. Let us check the logical flip:

In [28]:
print("Logical flip is ", int((logical_x @ r) % 2))

Logical flip is  1


Since the logical flip is $1$, the recovery fails.

In [29]:
print("Is the recovery successful: ", recovery(H_z, r, logical_x, H_x))

Is the recovery successful:  False
